In [ ]:
import pandas as pd
import numpy as np
import spacy
import re
import os
from math import ceil

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "lemmatizer"])


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving preprocessed_lung.csv to preprocessed_lung.csv


In [ ]:
df= pd.read_csv('preprocessed_lung.csv')

In [ ]:
df.columns

Index(['PMID', 'Title', 'Abstract', 'Authors', 'Affiliation', 'Country',
       'Journal', 'Year', 'DOI', 'URL', 'filtered_abstract', 'sentences',
       'tokens', 'final_tokens'],
      dtype='object')

In [ ]:
FINAL_DISEASE_NAMES = {
    "heart_disease": [
        "heart","heart disease","cardiovascular disease","cardiovascular diseases","cvd","coronary heart disease","coronary artery disease","ischemic heart disease", "atherosclerotic cardiovascular disease","ascvd","myocardial infarction","heart attack","acute coronary syndrome","angina pectoris","heart failure","congestive heart failure","hypertensive heart disease","peripheral artery disease","peripheral vascular disease","stroke","cerebrovascular disease","cardiac disease","vascular disease"
    ],

    "diabetes": [
        "diabetes", "diabetes mellitus", "type 1 diabetes","type i diabetes","t1d","type 2 diabetes","type ii diabetes","t2d","gestational diabetes","adult-onset diabetes", "juvenile diabetes","hyperglycemia", "chronic hyperglycemia","insulin resistance","impaired glucose tolerance","impaired fasting glucose","prediabetes","metabolic disease","metabolic disorder","glucose metabolism disorder", "diabetic disease"
    ],

    "lung_disease": [
        "lung disease",
        "chronic lung disease",
        "respiratory disease",
        "respiratory diseases",
        "chronic respiratory disease",
        "pulmonary disease",
        "pulmonary disorders",
        "chronic obstructive pulmonary disease",
        "copd",
        "asthma",
        "chronic bronchitis",
        "emphysema",
        "interstitial lung disease",
        "ild",
        "pulmonary fibrosis",
        "restrictive lung disease",
        "obstructive lung disease",
        "airway disease",
        "lower respiratory disease",
        "respiratory illness"
    ],

    "cancer": [
        "cancer",
        "cancers",
        "malignancy",
        "malignancies",
        "malignant neoplasm",
        "neoplasm",
        "neoplasms",
        "tumor",
        "tumors",
        "carcinoma",
        "adenocarcinoma",
        "squamous cell carcinoma",
        "sarcoma",
        "lymphoma",
        "leukemia",
        "solid tumor",
        "metastatic cancer",
        "advanced cancer",
        "oncologic disease",
        "oncological disease",

    "Adenocarcinoma", "Basal Cell Carcinoma", "Squamous Cell Carcinoma",
    "Renal Cell Carcinoma", "Transitional Cell Carcinoma", "Ductal Carcinoma In Situ",
    "Invasive Ductal Carcinoma", "Invasive Lobular Carcinoma", "Hepatocellular Carcinoma",
    "Adrenocortical Carcinoma", "Bronchioloalveolar Carcinoma", "Merkel Cell Carcinoma",
    "Sebaceous Carcinoma", "Acinar Cell Carcinoma", "Papillary Carcinoma",
    "Follicular Carcinoma", "Medullary Carcinoma", "Mucoepidermoid Carcinoma",

    "Osteosarcoma", "Chondrosarcoma", "Ewing Sarcoma", "Liposarcoma",
    "Rhabdomyosarcoma", "Leiomyosarcoma", "Angiosarcoma", "Kaposi Sarcoma",
    "Synovial Sarcoma", "Fibrosarcoma", "Malignant Fibrous Histiocytoma",
    "Alveolar Soft Part Sarcoma", "Clear Cell Sarcoma", "Desmoplastic Small Round Cell Tumor",
    "Epithelioid Sarcoma", "Gastrointestinal Stromal Tumor",

    "Acute Lymphoblastic Leukemia", "Acute Myeloid Leukemia",
    "Chronic Lymphocytic Leukemia", "Chronic Myeloid Leukemia",
    "Hairy Cell Leukemia", "Acute Promyelocytic Leukemia",
    "Chronic Myelomonocytic Leukemia", "Juvenile Myelomonocytic Leukemia",
    " Granular Lymphocytic Leukemia", "Blastic Plasmacytoid Dendritic Cell Neoplasm",

    "Hodgkin Lymphoma", "Non-Hodgkin Lymphoma", "Diffuse Large B-Cell Lymphoma",
    "Follicular Lymphoma", "Mantle Cell Lymphoma", "Burkitt Lymphoma",
    "Marginal Zone Lymphoma", "Primary Central Nervous System Lymphoma",
    "Primary Mediastinal B-Cell Lymphoma", "Anaplastic Large Cell Lymphoma",
    "Peripheral T-Cell Lymphoma", "Cutaneous T-Cell Lymphoma",
    "Mycosis Fungoides", "Sézary Syndrome", "Lymphoplasmacytic Lymphoma",
    "Primary Effusion Lymphoma", "Post-Transplant Lymphoproliferative Disorder",

    "Glioblastoma", "Astrocytoma", "Oligodendroglioma", "Ependymoma",
    "Medulloblastoma", "Meningioma", "Pituitary Adenoma", "Craniopharyngioma",
    "Schwannoma", "Pineoblastoma", "Pineocytoma", "Chordoma",
    "Central Neurocytoma", "Dysembryoplastic Neuroepithelial Tumor",

    "Neuroblastoma", "Retinoblastoma", "Wilms Tumor", "Rhabdoid Tumor",
    "Hepatoblastoma", "Germ Cell Tumor", "Osteosarcoma",
    "Ewing Sarcoma", "Rhabdomyosarcoma",

    "Ovarian Cancer", "Cervical Cancer", "Endometrial Cancer",
    "Uterine Cancer", "Vulvar Cancer", "Vaginal Cancer",
  "Choriocarcinoma",
    "Ovarian Germ Cell Tumor", "Fallopian Tube Cancer",

    "Prostate Cancer", "Testicular Cancer", "Bladder Cancer",
    "Kidney Cancer", "Penile Cancer", "Urethral Cancer",
    "Renal Pelvis Cancer", "Ureter Cancer",

    "Colorectal Cancer", "Colon Cancer", "Rectal Cancer",
    "Stomach Cancer", "Esophageal Cancer", "Pancreatic Cancer",
    "Liver Cancer", "Gallbladder Cancer", "Bile Duct Cancer",
    "Anal Cancer", "Small Intestine Cancer", "Appendiceal Cancer",
    "Peritoneal Cancer", "Gastroesophageal Junction Cancer",

    "Oral Cancer", "Tongue Cancer", "Throat Cancer",
    "Laryngeal Cancer", "Nasopharyngeal Cancer",
    "Oropharyngeal Cancer", "Hypopharyngeal Cancer",
    "Salivary Gland Cancer", "Paranasal Sinus Cancer",
    "Nasal Cavity Cancer", "Middle Ear Cancer",

    "Lung Cancer", "Small Cell Lung Cancer", "Non-Small Cell Lung Cancer",
    "Mesothelioma", "Thymoma", "Thymic Carcinoma",

    "Thyroid Cancer", "Adrenal Cancer", "Parathyroid Cancer",
    "Pituitary Cancer", "Pancreatic Neuroendocrine Tumor",
    "Carcinoid Tumor", "Pheochromocytoma",

    "Melanoma", "Skin Cancer (Non-Melanoma)",
    "Dermatofibrosarcoma Protuberans", "Cutaneous Lymphoma",

    "Bone Cancer", "Soft Tissue Sarcoma", "Giant Cell Tumor of Bone",

    "Multiple Myeloma", "Waldenström Macroglobulinemia",
    "Myelodysplastic Syndromes", "Myeloproliferative Neoplasms",
    "Polycythemia Vera", "Essential Thrombocythemia",
    "Primary Myelofibrosis", "Systemic Mastocytosis",

    "Desmoid Tumor", "Pseudomyxoma Peritonei",
    "Ameloblastoma", "Adamantinoma", "Extragonadal Germ Cell Tumor",
    "Phyllodes Tumor", "Male Breast Cancer", "Inflammatory Breast Cancer",
    "Triple-Negative Breast Cancer", "HER2-Positive Breast Cancer",
    "Metastatic Cancer", "Carcinoma of Unknown Primary"

    ]
}



In [ ]:


ENVIRONMENTAL_FACTORS = {
    "air_pollution": [
        "pm2.5", "pm10", "ultrafine particles", "ufp", "particulate matter",
        "nitrogen dioxide", "no2", "sulfur dioxide", "so2", "ozone", "o3",
        "carbon monoxide", "co", "volatile organic compounds", "vocs",
        "benzene", "formaldehyde", "toluene", "xylene", "ethylbenzene",
        "polycyclic aromatic hydrocarbons", "pahs", "dioxins", "furans",
        "black carbon", "elemental carbon", "organic carbon",
        "aerosol", "secondary inorganic aerosols",
        "air pollution", "diesel exhaust", "vehicle emissions",
        "industrial emissions", "power plant emissions", "dust",
        "road dust", "particles", "brake wear particles",
        "biomass burning", "wildfire smoke", "agricultural burning",
        "cigarette smoke", "secondhand smoke","tobacco smoke","airborne toxins", "airborne chemicals"
    ],

    "chemical_exposures": [
        "heavy metals", "lead", "pb", "mercury", "hg", "arsenic",
        "cadmium", "cd", "chromium", "cr", "nickel", "ni", "manganese",
        "aluminum",  "copper", "cu", "zinc", "zn", "selenium",
        "iron", "fe", "cobalt", "co", "vanadium", "antimony", "sb",
        "thallium", "tl", "beryllium",  "uranium",  "plutonium", "pu",
        "persistent organic pollutants", "pops", "polychlorinated biphenyls", "pcbs",
        "organochlorine pesticides", "ddt", "dieldrin", "chlordane", "heptachlor",
        "aldrin", "endrin", "toxaphene", "mirex", "hexachlorobenzene", "hcb",
        "perfluorinated compounds", "pfoa", "pfos", "pfas", "forever chemicals",
        "brominated flame retardants", "pbdes", "tbbpa", "hbcds",
        "phthalates", "dehp", "dbp", "bbp", "dibp", "dimp", "didp", "dinp",
        "bisphenols", "bpa", "bps", "bpf", "bpaf", "bpb", "bpe", "bpp",
        "parabens", "methylparaben", "ethylparaben", "propylparaben", "butylparaben",
        "triclosan", "triclocarban", "benzophenones", "oxybenzone", "avobenzone",
        "synthetic musks", "galaxolide", "tonalide", "cashmeran", "celestolide",
        "alkylphenols", "nonylphenol", "octylphenol", "bisphenol a", "bisphenol s",
        "solvents", "trichloroethylene", "tetrachloroethylene", "perchloroethylene",
        "methylene chloride", "chloroform", "carbon tetrachloride", "benzene",
        "toluene", "xylene", "styrene", "acetone", "ethanol", "methanol",
        "glycol ethers", "ethylene glycol", "propylene glycol",
        "plasticizers", "adipates", "citrates", "trimellitates", "sebacates",
        "antimicrobials", "quaternary ammonium compounds", "quats",
        "nanomaterials", "carbon nanotubes", "titanium dioxide nanoparticles",
        "silver nanoparticles", "quantum dots", "fullerenes", "graphene",
        "microplastics", "nanoplastics", "plastic particles", "synthetic fibers",
        "industrial chemicals", "phthalate esters", "alkylphenol ethoxylates",
        "polychlorinated naphthalenes", "chlorinated paraffins", "siloxanes",
        "acrylamide", "acrylonitrile", "vinyl chloride", "ethylene oxide",
        "propylene oxide", "butadiene", "isoprene", "formaldehyde", "acetaldehyde",
        "acrylic acid", "methacrylic acid", "maleic anhydride", "phthalic anhydride"
    ],

    "contaminants": [
       "trihalomethanes", "thms",
        "haloacetic acids", "haas", "chloroform", "bromodichloromethane",
        "dibromochloromethane", "bromoform", "chloramines", "chlorine dioxide",
        "ozonation byproducts", "bromate", "chlorite", "chlorate",
        "arsenic in water", "lead in water", "mercury in water",
        "cadmium in water", "chromium in water", "nitrate contamination",
        "nitrite contamination", "fluoride levels", "uranium in water",
        "radon in water", "perchlorate contamination", "pesticides in water",
        "herbicides in water", "insecticides in water", "fungicides in water",
        "pharmaceutical residues", "antibiotics in water", "hormones in water",
       "microplastics in water",
        "cyanotoxins", "microcystins", "cylindrospermopsin",
        "anatoxin-a", "saxitoxins", "bacterial contamination",
        "aflatoxins", "ochratoxin", "fumonisins", "zearalenone",
        "deoxynivalenol", "patulin", "mycotoxins", "marine biotoxins",
        "ciguatoxin", "saxitoxin", "domoic acid", "histamine",
        "food additives", "preservatives",
        "sodium nitrite", "sodium nitrate", "sulfites", "sulfur dioxide",
        "bha", "bht", "tbho", "propyl gallate", "benzoates", "sorbates",
        "acrylamide", "furan", "maillard reaction products",
        "heterocyclic amines", "polycyclic aromatic hydrocarbons in food",
        "advanced glycation end products",  "hydrogenated oils",
        "neotame", "advantame", "pfas from packaging","pfa"
        "styrene from containers", "vinyl chloride from packaging",
        "irradiated foods", "genetically modified foods", "gmo foods",
        "heavy metals", "lead", "arsenic",
        "cadmium", "chromium", "mercury",
        "pesticides", "herbicides", "pcbs",
        "dioxins", "pahs", "petroleum hydrocarbons",
        "btex compounds", "chlorinated", "radionuclides",
        "uranium", "radium", "thorium",
        "asbestos", "silica", "asbestos fibers",
        "soil erosion",
        "bioremediation", "phytoremediation", "soil vapor extraction",
        "soil amendments", "compost", "manure", "biosolids",
        "sewage sludge", "industrial sludge", "wastewater sludge", "geophagia", "pica", "soil contact",
        "dermal exposure", "inhalation exposure", "ingestion exposure"
    ],


    "built_environment": [
        "asbestos", "lead", "radon",
        "formaldehyde", "voc"
    ],

    "occupational_exposures": [
         "lead abatement", "mold remediation",
        "welding fumes",
        "dye exposure",
        "anesthetic gases", "chemotherapy drugs", "radiation exposure",
        "latex allergy", "needlestick injuries", "biological hazards",
    ],
    "radiation_exposures": [
        "ionizing radiation", "non-ionizing radiation", "radiation exposure",
        "natural radiation", "cosmic radiation", "terrestrial radiation",
        "radon gas", "radon exposure", "radon concentrations",
        "indoor radon", "basement radon", "soil radon",
        "radon mitigation", "radon testing", "radon remediation",
        "medical radiation", "diagnostic radiation", "x-rays",
        "ct scans", "computed tomography", "mammography",
        "fluoroscopy", "interventional radiology", "nuclear medicine",
        "radiation therapy", "radiotherapy", "brachytherapy",
        "occupational radiation", "radiation workers", "nuclear workers",
        "medical radiation workers", "dental radiation", "industrial radiography",
        "airline crew radiation", "cosmic radiation at altitude",
        "frequent flyer radiation", "space radiation", "solar radiation",
        "solar particle events", "galactic cosmic rays",
        "ultraviolet radiation", "uv radiation", "uv-a", "uv-b", "uv-c",
        "sunscreen chemicals",
         "heat radiation", "thermal radiation",
    ],




"other_FACTORS":[
    "traffic pollution", "roadway proximity", "diesel exhaust particles",
    "fine particulate matter", "pm2.5 exposure", "ozone exposure",
    "nitrogen oxides", "sulfur dioxide", "carbon monoxide",
    "indoor air pollution", "secondhand smoke", "environmental tobacco smoke",

    "arsenic exposure", "lead exposure", "cadmium exposure",
    "organophosphate pesticides", "phthalate exposure", "bpa exposure",
    "pfoa exposure", "pfc exposure", "persistent organic pollutants",


    "ionizing radiation", "medical radiation", "ultraviolet radiation",

    "chronic inflammation",
    "phthalates", "bisphenol a", "organotins", "organochlorine pesticides",
    "pcbs", "dioxins", "perfluorinated compounds", "polybrominated diphenyl ethers",
    "triclosan", "parabens", "benzophenones",
    "pm2.5", "ozone", "nitrogen dioxide", "traffic-related pollution",

    "arsenic in water", "nitrate in water", "fluoride in water",
    "pm2.5", "pm10", "ozone", "nitrogen dioxide", "sulfur dioxide",
    "traffic pollution", "diesel exhaust", "indoor air pollution",
    "secondhand smoke", "biomass smoke", "wildfire smoke",
    "mold spores", "dust mites", "cockroach allergens", "pet dander",
    "pollen", "endotoxins", "volatile organic compounds",
    "asbestos", "silica dust", "coal dust",
    "isocyanates",

    "radon gas", "indoor radon", "basement radon",

    "benzene", "formaldehyde", "arsenic", "chromium vi",
    "nickel compounds", "cadmium", "beryllium", "vinyl chloride",
    "ethylene oxide", "acrylamide", "acrylonitrile",
    "polycyclic aromatic hydrocarbons", "heterocyclic amines",
    "nitrosamines", "aflatoxins", "pcbs", "dioxins",
    "organochlorine pesticides", "chlorinated solvents",

    "pm2.5", "diesel exhaust", "outdoor air pollution",
    "indoor air pollution", "secondhand smoke", "radon gas",

    "ionizing radiation", "ultraviolet radiation", "medical radiation",
    "radon exposure", "gamma radiation", "x-rays",
    "radiofrequency radiation", "extremely low frequency fields",
    "tobacco use",
    "arsenic in water", "chlorination byproducts", "nitrate in water",
    "radon in water", "industrial contaminants in water"
],
    "hazardous_substances": [
    "aromatic amines",
    "Asbestos", "asbestiform fibres",
    "volatile organic compounds",
    "nitrogen oxides",
    "fine particulates",
    "occupational exposure",
    "radon",
    "passive smoking",
    "sulphuric acid",
    "inorganic arsenic",
    "SO2",
    "benzene",
    "residential radon exposure",
    "cooking oil vapours",
    "chlorine",
    "hypochlorite",
    "chloramine",
    "chlorine dioxine",
    "ozone",
    "Trihalomethanes", "chloroform", "bromodichloromethane", "chlorodibromomethane", "bromoform",
    "Brominated by-products",
    "nitrites & nitrates",
    "radionuclides",
    "formaldehyde",
    "pyrene",
    "fluoranthene",
    "naphthalenes and catechol",
    "Nickel",
    "cadmium",
    "cobalt",
    "Vinyl chloride",
    "Aflatoxin",
    "aromatic amines",
    "ammonia",
    "tar",
    "carbon monoxide",
    "nicotine",
    "trichloromethane",
    "styrene",
    "trichloroethylene",
    "tetrachloroethylene",
    "carbon tetrachloride",
    "chloroform",
    "Organochlorines",
    "creosote",
    "sulfallate",
    "Beryllium",
    "Chromium",
    "Ethylene oxide",
    "Smoke",
    "Gasoline",
    "Soot",
    "Ionizing radiation",
    "Noise",
    "O3",
    "NO2",
    "volatile organic compounds",
    "benzene",
    "CO",
    "SO2",
    "ultrafine particles",
    "light pollution",
    "desert dust",
    "wildfires",
    "volcano eruptions",
    "air pollution",
    "arsenic",
    "cadmium",
    "lead",
    "mercury",
    "tungsten",
    "antimony",
    "biomass fuels",
    "secondhand smoke",
    "extreme temperature",
    "tobacco",
    "Perfluoroalkyl",
    "Polyfluoro",
    "perfluoro",
    "mercury",
    "Pesticides",
    "lead",
    "polycyclic aromatic hydrocarbons",
    "cadmium",
    "chromium",
    "arsenic",
    "diisocyanates",
    "Tobacco",
    "nitrogen oxide",
    "sulfur dioxide",
    "carbon monoxide",
    "volatile organic compounds",
    "particulate matter",
    "NO2",
    "Coal dust",
    "Livestock farming and agriculture",
    "biological dusts",
    "mineral dusts",
    "gases or fumes",
    "cadmium",
    "barium",
    "cobalt",
    "molybdenum",
    "mercury",
    "cesium",
    "manganese",
    "antimony",
    "lead",
    "tin",
    "strontium",
    "tungsten",
    "thallium",
    "uranium",
    "arsenous acid",
    "arsenic acid",
    "arsenobetaine",
    "arsenocholine",
    "dimethylarsinic acid",
    "monomethylarsonic acid",
    "total arsenic",
    "1-hydroxynaphthalene",
    "2-hydroxynaphthalene",
    "3-hydroxyfluorene",
    "2-hydroxyfluorene",
    "1-hydroxyphenanthrene",
    "1-hydroxypyrene",
    "2 & 3-hydroxyphenanthrene",
    "sulfates",
    "ammonia",
    "sodium chloride",
    "black carbon",
    "mineral dust",
    "Pesticides",
    "dichlorodiphenyltrichloroethane",
    "lindane chlordane",
    "chlorinated hydrocarbons",
    "polycyclic aromatic hydrocarbons",
    "Bisphenol A",
    "Phthalates",
    "butyl benzyl phthalate",
    "Polybrominated Diphenyl Ethers",
    "Arsenic",
    "Cadmium",
    "nitrates",
    "nitrites",
    "N-nitroso",
    "N-nitroso",
    "arsenic",
    "dioxins",
    "talc",
    "straight oil machining fluids",
    "nicotine",
    "dichlorodiphenyldichloroethylene"
]
}


In [ ]:
contribution_phrases = [
    "constitute a major contributor to",
    "represent a major source of",
    "act as a primary contributor to",
    "serve as a dominant source of",
    "account for a significant proportion of",
    "play a central role in",
    "pose a substantial source of",
    "are a leading contributor to",
    "function as a key driver of",
    "emerge as a major determinant of",
    "are a principal source of",
    "are the main driver of",
    "constitute the primary source of",
    "are fundamental to",
    "underpin a large fraction of",
    "are chiefly responsible for",
    "are a predominant factor in",
    "explain a major share of",
    "are instrumental in",
    "are a critical component of",
    "are a decisive factor in",
    "lie at the heart of",
    "are the cornerstone of",
    "are largely attributable to",
    "are the most significant contributor to",

    "contribute significantly to",
    "serve as an important contributor to",
    "play an important role in",
    "are a key factor in",
    "have a substantial impact on",
    "act as an important source of",
    "are a notable contributor to",
    "represent an important pathway for",
    "influence exposure pathways related to",
    "are a relevant source of",
    "make a significant contribution to",
    "are a considerable source of",
    "exert a strong influence on",
    "are a material contributor to",
    "are a prominent factor in",
    "are a salient contributor to",
    "are a meaningful source of",
    "are a non-negligible contributor to",
    "are a consistent predictor of",
    "contribute materially to",
    "are an established factor in",
    "are a recognized source of",
    "are a non-trivial source of",
    "are associated with",
    "are linked to",
    "are correlated with",
    "are connected to",
    "are related to",
    "show an association with",
    "demonstrate a relationship with",
    "are implicated in",
    "are involved in",
    "contribute to processes related to",
    "co-vary with",
    "exhibit a connection to",
    "present a linkage to",
    "demonstrate an association with",
    "share a relationship with",
    "are a component of",
    "are a facet of",
    "are a part of",
    "are a characteristic of",
    "are a feature of",
    "are a parameter in",
    "are a variable associated with",
    "may contribute to",
    "may play a role in",
    "may represent a source of",
    "have been suggested as a contributor to",
    "are thought to contribute to",
    "appear to influence",
    "could act as a source of",
    "are potentially involved in",
    "may account for part of",
    "are suspected contributors to",
    "might contribute to",
    "could potentially be a source of",
    "are hypothesized to contribute to",
    "have been proposed as a contributor to",
    "are a possible source of",
    "are a putative contributor to",
    "are a candidate for",
    "are plausibly linked to",
    "are conjectured to be involved in",
    "are tentatively associated with",
    "could partly explain",
    "are not ruled out as a source of",
    "are increasingly recognized as",
    "are receiving growing attention as",
    "are emerging as an important factor in",
    "remain underexplored despite contributing to",
    "have been largely overlooked as a source of",
    "are insufficiently studied contributors to",
    "represent a neglected pathway for",
    "constitute an understudied source of",
    "have received limited attention as",
    "are poorly characterized contributors to",
    "are a nascent but important source of",
    "are a hidden contributor to",
    "are an unrecognized pathway for",
    "are a blind spot in understanding",
    "represent a non-conventional source of",
    "are a non-traditional contributor to",
    "constitute a legacy source of",
    "are a re-emerging concern for",
    "are a poorly quantified source of",
    "are an unresolved source of",
    "present an unaddressed challenge for",
    "are a data gap in assessing",

    "contribute to radiological exposure pathways",
    "influence environmental contamination routes",
    "act as a pathway for human exposure",
    "serve as a vector for environmental exposure",
    "affect population-level exposure risks",
    "contribute to chronic exposure mechanisms",
    "impact inhalation and ingestion pathways",
    "alter environmental exposure dynamics",
    "increase long-term exposure potential",
    "shape indirect exposure routes",
    "mediate the relationship between X and Y",
    "modulate the levels of",
    "govern the distribution of",
    "control the fate and transport of",
    "are a mechanism for",
    "facilitate the mobilization of",
    "are a conduit for",
    "are a reservoir for",
    "are a sink and source for",
    "drive the cycling of",
    "regulate the flux of",
    "are integral to the pathway of",
    "elevate the risk of",
    "amplify the hazards associated with",
    "exacerbate the problem of",
    "compound existing burdens of",
    "are a risk multiplier for",
    "increase the vulnerability to",
    "are a threat multiplier regarding",
    "heighten susceptibility to",
    "intensify the impacts of",
    "worsen the outcomes of",

    "enhance the effect",
    "increase efficacy",
    "associated with",
    "linked to",
    "related to",
    "increased",
    "elevated",
    "higher",
    "higher probabilistic",
    "increased risk",
    "significant risk factors",
    "significant association",
    "can lead to",
    "result in",
    "cause",
    "deleterious role of",
    "detrimental effect",
    "adverse impact",
    "was elevated",
    "showed an increase",
    "demonstrated higher rates",
    "linear",
    "dose-response",
    "proportional",
    "severe adverse health effects",
    "significant morbidity",

    "amplify the impact",
    "augment the response",
    "boost effectiveness",
    "potentiate the outcome",
    "heighten the result",
    "synergize with",
    "act synergistically",
    "improve efficacy",
    "intensify the effect",
    "correlated with",
    "connected to",
    "tied to",
    "accompanied by",
    "co-occurring with",
    "concurrent with",
    "attributable to",
    "a predictor of",
    "raised levels of",
    "augmented",
    "amplified",
    "heightened",
    "excess",
    "greater",
    "more",
    "upregulated",
    "accentuated",
    "escalated",
    "elevated risk",
    "elevated probability",
    "greater likelihood",
    "greater chance",
    "greater odds",
    "increased hazard",
    "heightened susceptibility",
    "heightened vulnerability",
    "excess relative risk",
    "elevated odds ratio",
    "elevated hazard ratio",
    "elevated standardized incidence ratio",
    "strong predictor",
    "strong determinant",
    "major risk factor",
    "key risk factor",
    "important risk factor",
    "statistically significant link",
    "statistically significant correlation",
    "robust association",
    "clear contributor",
    "independent risk factor",
    "contribute to",
    "give rise to",
    "precipitate",
    "trigger",
    "induce",
    "promote",
    "drive",
    "foster",
    "predispose to",
    "harmful effect of",
    "injurious effect of",
    "damaging role of",
    "negative impact",
    "negative influence",
    "detrimental association with",
    "contributes detrimentally to",
    "worsens",
    "exacerbates",
    "exhibited increased",
    "displayed elevated",
    "showed a rise in",
    "demonstrated higher",
    "presented with excess",
    "was heightened",
    "registered an increase in",
    "dose-dependent increase",
    "dose-dependent effect",
    "proportional relationship",
    "gradient of risk",
    "monotonic increase",
    "direct correlation",
    "scaling with exposure",
    "serious health consequences",
    "major health consequences",
    "substantial detrimental outcomes",
    "profound negative health impacts",
    "marked increase in disease",
    "marked increase in mortality",
    "major contributor to morbidity",
    "significant health hazard",
    "significant health threat",

    # Statistical/measurement terms
    "odds ratio",
    "hazard ratio",
    "relative risk",
    "standardized incidence ratio",
    "risk ratio",
    "excess risk",
    "attributable risk",
    "positive correlation",
    "positive association",
    "statistically significant",
    "p < 0.05",
    "confidence interval excludes 1",
    "dose-response relationship",
    "exposure-response gradient",
    "elevates",
    "raises",
    "boosts",
    "heightens",
    "intensifies",
    "amplifies",
    "augments",
    "potentiates",
    "exacerbates",
    "aggravates",
    "worsens",
    "promotes",
    "facilitates",
    "accelerates",
    "induces",
    "triggers",
    "provokes",
    "stimulates",
    "more likely",
    "greater risk",
    "higher incidence",
    "increased prevalence",
    "elevated mortality",
    "excess mortality",
    "higher rates",
    "greater frequency",
    "increased susceptibility",
    "enhanced vulnerability"

]

In [ ]:
hazardous_substances = [
    "aromatic amines",
    "Asbestos", "asbestiform fibres",
    "volatile organic compounds",
    "nitrogen oxides",
    "fine particulates",
    "occupational exposure",
    "radon",
    "passive smoking",
    "sulphuric acid",
    "inorganic arsenic",
    "SO2",
    "benzene",
    "residential radon exposure",
    "cooking oil vapours",
    "chlorine",
    "hypochlorite",
    "chloramine",
    "chlorine dioxine",
    "ozone",
    "Trihalomethanes", "chloroform", "bromodichloromethane", "chlorodibromomethane", "bromoform",
    "Brominated by-products",
    "nitrites & nitrates",
    "radionuclides",
    "formaldehyde",
    "pyrene",
    "fluoranthene",
    "naphthalenes and catechol",
    "Nickel",
    "cadmium",
    "cobalt",
    "Vinyl chloride",
    "Aflatoxin",
    "aromatic amines",
    "ammonia",
    "tar",
    "carbon monoxide",
    "nicotine",
    "trichloromethane",
    "styrene",
    "trichloroethylene",
    "tetrachloroethylene",
    "carbon tetrachloride",
    "chloroform",
    "Organochlorines",
    "creosote",
    "sulfallate",
    "mycotoxins",
    "ergotism",
    "Beryllium",
    "Chromium",
    "Ethylene oxide",
    "Smoke",
    "Gasoline",
    "Soot",
    "Ionizing radiation",
    "Noise",
    "O3",
    "NO2",
    "volatile organic compounds",
    "benzene",
    "CO",
    "SO2",
    "ultrafine particles",
    "light pollution",
    "desert dust",
    "wildfires",
    "volcano eruptions",
    "air pollution",
    "arsenic",
    "cadmium",
    "lead",
    "mercury",
    "tungsten",
    "antimony",
    "biomass fuels",
    "secondhand smoke",

    "tobacco",
    "Perfluoroalkyl",
    "Polyfluoro",
    "perfluoro",
    "mercury",
    "Pesticides",
    "lead",
    "polycyclic aromatic hydrocarbons",
    "cadmium",
    "chromium",
    "arsenic",
    "diisocyanates",
    "Tobacco",
    "nitrogen oxide",
    "sulfur dioxide",
    "carbon monoxide",
    "volatile organic compounds",
    "particulate matter",
    "NO2",
    "gases or fumes",
    "cadmium",
    "barium",
    "cobalt",
    "molybdenum",
    "mercury",
    "cesium",
    "manganese",
    "antimony",
    "lead",
    "tin",
    "strontium",
    "tungsten",
    "thallium",
    "uranium",
    "arsenous acid",
    "arsenic acid",
    "arsenobetaine",
    "arsenocholine",
    "dimethylarsinic acid",
    "monomethylarsonic acid",
    "total arsenic",
    "1-hydroxynaphthalene",
    "2-hydroxynaphthalene",
    "3-hydroxyfluorene",
    "2-hydroxyfluorene",
    "1-hydroxyphenanthrene",
    "1-hydroxypyrene",
    "2 & 3-hydroxyphenanthrene",
    "sulfates",
    "ammonia",
    "sodium chloride",
    "black carbon",
    "mineral dust",
    "Pesticides",
    "dichlorodiphenyltrichloroethane",
    "lindane chlordane",
    "chlorinated hydrocarbons",
    "polycyclic aromatic hydrocarbons",
    "Bisphenol A",
    "Phthalates",
    "butyl benzyl phthalate",
    "Polybrominated Diphenyl Ethers",
    "Arsenic",
    "Cadmium",
    "nitrates",
    "nitrites",
    "N-nitroso",
    "N-nitroso",
    "arsenic",
    "dioxins",
    "talc",
    "straight oil machining fluids",
    "nicotine",
    "dichlorodiphenyldichloroethylene"
]

In [ ]:
print(ENVIRONMENTAL_FACTORS.keys())

dict_keys(['air_pollution', 'chemical_exposures', 'contaminants', 'built_environment', 'occupational_exposures', 'radiation_exposures', 'other_FACTORS', 'hazardous_substances'])


In [ ]:
# annotated_df.to_csv("an.csv", index=False, encoding="utf-8")


In [ ]:
import pandas as pd
import spacy
import re
import os
from math import ceil
from tqdm import tqdm  # Progress bar


nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "lemmatizer"])


In [ ]:
def build_lookup(dictionary):
    lookup = {}
    for tag, phrases in dictionary.items():
        for phrase in phrases:
            lookup[phrase.lower()] = tag
    return lookup

DISEASE_LOOKUP = build_lookup(FINAL_DISEASE_NAMES)
ENV_LOOKUP = build_lookup(ENVIRONMENTAL_FACTORS)

DISEASE_PHRASES = sorted(DISEASE_LOOKUP.keys(), key=len, reverse=True)
ENV_PHRASES = sorted(ENV_LOOKUP.keys(), key=len, reverse=True)

In [ ]:
def find_spans(text, phrases):
    spans = []
    for phrase in phrases:
        pattern = rf"\b{re.escape(phrase)}\b"
        for m in re.finditer(pattern, text, flags=re.IGNORECASE):
            spans.append((m.start(), m.end(), phrase))
    return spans

In [ ]:
def annotate_sentence(sentence):
    doc = nlp(sentence)
    tokens = [t.text for t in doc]
    tags = ["O"] * len(tokens)

    for start, end, phrase in find_spans(sentence, DISEASE_PHRASES):
        idxs = [
            i for i, tok in enumerate(doc)
            if tok.idx >= start and tok.idx + len(tok.text) <= end
        ]
        if not idxs:
            continue
        disease_tag = DISEASE_LOOKUP[phrase]
        tags[idxs[0]] = f"B-{disease_tag}"
        for i in idxs[1:]:
            tags[i] = f"I-{disease_tag}"
    for start, end, phrase in find_spans(sentence, ENV_PHRASES):
        idxs = [
            i for i, tok in enumerate(doc)
            if tok.idx >= start and tok.idx + len(tok.text) <= end
        ]
        if not idxs:
            continue
        factor_tag = ENV_LOOKUP[phrase]
        for i in idxs:
            if tags[i] == "O":
                tags[i] = f"B-{factor_tag}" if i == idxs[0] else f"I-{factor_tag}"

    return tokens, tags

In [ ]:

annotated_rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Annotating sentences"):
    sentence = row["sentences"]
    tokens, tags = annotate_sentence(sentence)
    annotated_rows.append({

        "PMID": row.get("PMID", ""),
        "authors": row.get("Authors", ""),
        "affiliation": row.get("Affiliation", ""),
        "country": row.get("Country", ""),
        "journal": row.get("Journal", ""),
        "year": row.get("Year", ""),
        "title": row.get("Title", ""),
        "abstract": row.get("Abstract", ""),
        "filtered_abstract": row.get("filtered_abstract", ""),
         "sentence": sentence,
        "tokens": tokens,
        "tags": tags,
    })

annotated_df = pd.DataFrame(annotated_rows)

num_subsets = 10
subset_size = ceil(len(annotated_df) / num_subsets)
output_dir = "annotated_subsets"
os.makedirs(output_dir, exist_ok=True)

for i in tqdm(range(num_subsets), desc="Saving subsets"):
    start_idx = i * subset_size
    end_idx = min((i + 1) * subset_size, len(annotated_df))
    subset_df = annotated_df.iloc[start_idx:end_idx]

    subset_filename = os.path.join(output_dir, f"annotated_subset_{i+1}.csv")
    subset_df.to_csv(subset_filename, index=False)


Saving subsets: 100%|██████████| 10/10 [00:04<00:00,  2.11it/s]


In [ ]:
annotated_df.to_csv("ner-lung.csv", index=False, encoding="utf-8")


In [ ]:
from google.colab import files

files.download("ner-lung.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>